In [0]:
from pyspark.sql import functions as F

AUDIT_TABLE = "atliq.silver.audit_log"

dbutils.widgets.text("status", "SUCCESS")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("job_run_id", "")
dbutils.widgets.text("run_date", "")

status = dbutils.widgets.get("status")
job_name = dbutils.widgets.get("job_name")
job_run_id = dbutils.widgets.get("job_run_id")
run_date = dbutils.widgets.get("run_date")

if status not in ("SUCCESS", "FAILED"):
    raise ValueError(f"status must be SUCCESS or FAILED, got '{status}'")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
  audit_ts TIMESTAMP, run_date STRING, job_name STRING, job_run_id STRING,
  status STRING, orders_rows BIGINT, payments_rows BIGINT, order_items_rows BIGINT)
""")

DataFrame[]

In [0]:
def safe_count(table):
    try:
        return spark.table(table).count()
    except Exception:
        return None

orders_n = safe_count("atliq.silver.orders")
payments_n = safe_count("atliq.silver.payments")
items_n = safe_count("atliq.silver.order_items")

In [0]:
spark.sql(f"DELETE FROM {AUDIT_TABLE} WHERE job_run_id = '{job_run_id}' AND status = '{status}'")

row = [(run_date, job_name, job_run_id, status, orders_n, payments_n, items_n)]
(spark.createDataFrame(row,
        "run_date STRING, job_name STRING, job_run_id STRING, status STRING, "
        "orders_rows BIGINT, payments_rows BIGINT, order_items_rows BIGINT")
      .withColumn("audit_ts", F.current_timestamp())
      .select("audit_ts", "run_date", "job_name", "job_run_id", "status",
              "orders_rows", "payments_rows", "order_items_rows")
      .write.mode("append").saveAsTable(AUDIT_TABLE))

print(f"AUDIT | {job_name} | {job_run_id} | {status} | orders={orders_n} payments={payments_n} items={items_n}")

AUDIT | manual_test | test1 | SUCCESS | orders=10020 payments=8341 items=23210
